# Capstone: Unlearn a Data Subject, Synthesise Replacement

## What This Is
This capstone combines machine unlearning and synthetic data generation into a complete GDPR compliance pipeline:

1. **Receive deletion request**: A data subject exercises their right to erasure
2. **Audit the model**: Verify the subject's data is present (membership inference)
3. **Unlearn**: Apply approximate unlearning to remove their influence
4. **Verify unlearning**: MIA confirms the model no longer recognizes the subject as a member
5. **Synthesise replacement**: Generate a synthetic record to maintain dataset statistics
6. **Re-evaluate**: Confirm model performance is preserved after the operation

This pipeline demonstrates that privacy compliance and model utility can coexist.

In [ ]:
import numpy as np
import json
from typing import Dict, List, Tuple, Optional

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_lr(X, y, lr=0.05, epochs=300):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(epochs):
        e = sigmoid(X @ w + b) - y
        w -= lr * (X.T @ e) / len(y); b -= lr * e.mean()
    return w, b

def predict(X, w, b): return (sigmoid(X @ w + b) > 0.5).astype(int)
def acc(X, y, w, b): return (predict(X, y, w, b) == y).mean() if len(y) > 0 else 0.0

# Wait, fix acc signature
def model_acc(X, y, w, b): return ((sigmoid(X @ w + b) > 0.5).astype(int) == y).mean()

# Build credit scoring dataset
N = 600
age = np.random.normal(40, 12, N).clip(18, 80)
income = (20000 + age * 1000 + np.random.normal(0, 8000, N)).clip(15000, 200000)
credit = (500 + income / 1000 + np.random.normal(0, 50, N)).clip(300, 850)
loan = (income * 0.3 * np.random.uniform(0.5, 2.0, N)).clip(1000, 100000)
default = ((loan / income > 0.4) & (credit < 650)).astype(float)

X = np.column_stack([age, income, credit, loan])
y = default

# Train/test split
X_train, y_train = X[:500], y[:500]
X_test, y_test = X[500:], y[500:]

# Normalize
mu, sigma = X_train.mean(0), X_train.std(0) + 1
X_train_n = (X_train - mu) / sigma
X_test_n = (X_test - mu) / sigma

# Train original model
w_orig, b_orig = train_lr(X_train_n, y_train)
print(f'Original model accuracy: {model_acc(X_test_n, y_test, w_orig, b_orig):.3f}')


In [ ]:
# Step 1: Identify data subject requesting deletion
# Subject: record index 42 (an individual who defaults)

FORGET_IDX = [42, 43, 44]  # 3 records from the same individual (multiple transactions)

# Step 2: Audit — verify subject IS a member (membership inference)
def compute_loss(X, y, w, b):
    z = X @ w + b
    p = np.clip(sigmoid(z), 1e-10, 1 - 1e-10)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

subject_X = X_train_n[FORGET_IDX]
subject_y = y_train[FORGET_IDX]
subject_loss = compute_loss(subject_X, subject_y, w_orig, b_orig).mean()

# Compare to average non-member loss
rand_test_idx = np.random.choice(len(X_test_n), 50, replace=False)
test_loss = compute_loss(X_test_n[rand_test_idx], y_test[rand_test_idx], w_orig, b_orig).mean()

print('Step 1-2: Membership Audit')
print(f'  Subject average loss:    {subject_loss:.4f}')
print(f'  Non-member average loss: {test_loss:.4f}')
verdict = 'MEMBER (deletion required)' if subject_loss < test_loss else 'NON-MEMBER'
print(f'  Verdict: {verdict}')


In [ ]:
# Step 3: Apply unlearning
retain_idx = [i for i in range(len(X_train_n)) if i not in FORGET_IDX]
X_retain_n = X_train_n[retain_idx]
y_retain = y_train[retain_idx]

def gradient_ascent_unlearn(w0, b0, Xf, yf, Xr, yr, lr_a=0.05, lr_d=0.03, epochs=40):
    w, b = w0.copy(), b0
    for _ in range(epochs):
        ef = sigmoid(Xf @ w + b) - yf
        er = sigmoid(Xr @ w + b) - yr
        dw_f = (Xf.T @ ef) / len(yf)
        dw_r = (Xr.T @ er) / len(yr)
        w = w + lr_a * dw_f - lr_d * dw_r
        b = b + lr_a * ef.mean() - lr_d * er.mean()
    return w, b

w_unlearn, b_unlearn = gradient_ascent_unlearn(
    w_orig, b_orig, subject_X, subject_y, X_retain_n, y_retain
)

# Step 4: Verify unlearning
subject_loss_after = compute_loss(subject_X, subject_y, w_unlearn, b_unlearn).mean()
test_loss_after = compute_loss(X_test_n[rand_test_idx], y_test[rand_test_idx], w_unlearn, b_unlearn).mean()

print('Step 3-4: Unlearning Verification')
print(f'  Subject loss before unlearning: {subject_loss:.4f}')
print(f'  Subject loss after unlearning:  {subject_loss_after:.4f}')
print(f'  Non-member loss after:          {test_loss_after:.4f}')
gap = subject_loss_after - test_loss_after
verdict2 = 'ERASED (gap near 0)' if abs(gap) < 0.1 else 'NOT FULLY ERASED'
print(f'  Gap: {gap:.4f} → {verdict2}')

print(f'\n  Model accuracy preserved: {model_acc(X_test_n, y_test, w_unlearn, b_unlearn):.3f} '
      f'(orig: {model_acc(X_test_n, y_test, w_orig, b_orig):.3f})')


In [ ]:
# Step 5: Synthesise replacement records
# Generate synthetic records with similar statistical properties to the deleted ones

def synthesise_replacement(deleted_X: np.ndarray, training_X: np.ndarray, n: int) -> np.ndarray:
    """Generate synthetic records near the deleted records but with added noise."""
    mu_del = deleted_X.mean(axis=0)
    # Use training set variance for realistic noise
    std_train = training_X.std(axis=0)
    synthetic = mu_del + np.random.randn(n, deleted_X.shape[1]) * std_train * 0.3
    return synthetic

X_replacement = synthesise_replacement(subject_X, X_retain_n, len(FORGET_IDX))

# Assign labels using the unlearned model
y_replacement = (sigmoid(X_replacement @ w_unlearn + b_unlearn) > 0.5).astype(float)

print('Step 5: Synthetic Replacement Generation')
print(f'  Deleted records:           {len(FORGET_IDX)}')
print(f'  Synthetic replacements:    {len(X_replacement)}')
print(f'  Original subject labels:   {subject_y}')
print(f'  Synthetic labels:          {y_replacement}')

# Final report
print('\n' + '=' * 55)
print('GDPR Deletion Pipeline — Final Report')
print('=' * 55)
report = {
    'subject_records_deleted': len(FORGET_IDX),
    'unlearning_method': 'gradient_ascent',
    'pre_unlearn_member_loss': round(float(subject_loss), 4),
    'post_unlearn_member_loss': round(float(subject_loss_after), 4),
    'model_acc_preserved': round(float(model_acc(X_test_n, y_test, w_unlearn, b_unlearn)), 3),
    'synthetic_replacements_generated': len(X_replacement),
    'compliance_status': 'COMPLIANT' if abs(gap) < 0.15 else 'REQUIRES_REVIEW',
}
for k, v in report.items():
    print(f'  {k}: {v}')
